# **Run LLM Falcon Locally**

### **Objective:**  
To learn how to set up and run the **Falcon rw 1b** large language model (LLM) locally using Hugging Face and LangChain. This demo demonstrates downloading the Falcon model, configuring it for inference, creating a custom conversation chain, and interacting with it using memory and prompt templates.

---

### **Note:**  
- Before running any demo, ensure that the **requirements.txt** file is installed. This file contains all the required dependencies for **all demos and guided practices under Building LLM Applications**.
- If the dependencies were already installed earlier (after creating the virtual environment), there is no need to install them again. You can directly proceed with running the demo.
- Refer to Lesson_01 **Demo_01_Zero_Shot_Prompting.ipynb** Step 1 for creating a virtual environment and installing the requirements.txt 
- Ensure you select the right kernel **Python (myenv)** while running the demos
---


### **Steps to perform:**
1. Set up the environment  
2. Download the Falcon rw 1b model and tokenizer from Hugging Face  
3. Set up model and generation configuration  
4. Build the conversation chain  
5. Modify the prompt template to define a specific conversational style  
6. Manage conversation history with ConversationBufferWindowMemory  
7. Interact with the LLM  

---


### **Step 1: Set up the Environment**
- Import all required libraries for model loading, prompt templates, and generation pipelines

In [1]:
import warnings; warnings.filterwarnings('ignore')
import torch
from langchain_core.prompts import PromptTemplate
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


- Ensure your environment has GPU enabled to optimize Falcon model inference speed.

### **Step 2: Download the Falcon rw 1b model and tokenizer from Hugging Face**
- Load the Falcon rw 1b Instruct model and tokenizer from Hugging Face’s tiiuae repository

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "tiiuae/falcon-rw-1b"


# Configure 8-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True  # allows partial offloading to CPU if GPU is full
)

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",             # automatically split layers between GPU/CPU
    quantization_config=bnb_config # replaces load_in_8bit=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model.eval()

print(f"✅ Falcon model loaded on device(s): {model.device}")




The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model device: cuda:0


### **Step 3: Set up model and generation configuration**
- Configure model generation parameters such as token limits, temperature, and repetition penalty for natural and varied responses

In [3]:
from langchain_huggingface import HuggingFacePipeline 
# Step 3: Generation config (EXACT original)
generation_config = model.generation_config
generation_config.temperature = 0.9
generation_config.num_return_sequences = 1
generation_config.max_new_tokens = 256
generation_config.use_cache = False
generation_config.repetition_penalty = 1.7
generation_config.pad_token_id = tokenizer.eos_token_id
generation_config.eos_token_id = tokenizer.eos_token_id

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)
llm_pipeline = HuggingFacePipeline(pipeline=pipe)



Device set to use cuda:0


- The pipeline integrates Falcon with LangChain, allowing text generation and conversational chaining.

### **Step 4: Build the conversation chain**
- Define an initial prompt and structure the dialogue between a human and the AI model




In [4]:

# Step 4: Original initial_prompt
initial_prompt = """
The following is a conversation between a human and an AI. The AI is knowledgeable and provides detailed answers.

Current conversation:

Human: What is the theory of relativity?
AI:
""".strip()

dialogue_chain_prompt = "Human: {input}\nAI: {chat_history}"
print("Step 4:", dialogue_chain_prompt)


Step 4: Human: {input}
AI: {chat_history}


- The prompt sets the conversation context for the model, simulating a realistic dialogue flow.

### **Step 5: Modify the prompt template to define a specific conversational style**
- Customize the prompt to make the AI adopt a particular persona, in this case, Albert Einstein

In [5]:

# Step 5: Original Einstein template
new_template = """
The following is a conversation between a human and an AI. The AI behaves like Albert Einstein, providing detailed explanations about physics.

Current conversation:
{history}
Human: {input}
AI:""".strip()

prompt = PromptTemplate(input_variables=["history", "input"], template=new_template)
print("Step 5:", new_template)



Step 5: The following is a conversation between a human and an AI. The AI behaves like Albert Einstein, providing detailed explanations about physics.

Current conversation:
{history}
Human: {input}
AI:


- This allows you to tailor the LLM’s tone, depth, and expertise dynamically through LangChain’s prompt engineering capabilities.

### **Step 6: Manage conversation history with ConversationBufferWindowMemory**
- Implement a simple memory system that keeps track of the last few conversation turns

In [6]:
# Step 6: Manual memory (k=6)
history = []

def predict(input_text):
    hist = "\n".join([f"Human: {h}\nAI: {a}" for h, a in history[-6:]])
    full_prompt = new_template.format(history=hist or "", input=input_text)
    response = llm_pipeline.invoke(full_prompt).strip()
    history.append((input_text, response))
    return response


- This manual memory buffer retains the last six exchanges, ensuring contextual continuity during conversation without excessive memory usage.

# **Step 7: Interact with the LLM**
- Test the locally running Falcon LLM with a conversational prompt

In [7]:
# Step 7: Original demo
text = "Suggest me a company name for car manufacturing using V8 engine."
res = predict(text)
print("Step 7:", res)


Step 7: The following is a conversation between a human and an AI. The AI behaves like Albert Einstein, providing detailed explanations about physics.

Current conversation:

Human: Suggest me a company name for car manufacturing using V8 engine.
AI: How about "V8 Auto?"
User


# **Conclusion**
By following this tutorial, you have successfully learned how to run Falcon 7B locally and integrate it with LangChain’s PromptTemplate and ConversationChain.
You explored how to:
- Configure Falcon for efficient inference,
- Create customized conversational prompts, and
- Manage dialogue context with memory buffers.

This workflow provides the foundation for building local, private, and customizable LLM-based assistants without relying on cloud APIs.

---